# Final Descriptive Statistics

This notebook freezes the final thesis sample and regenerates the general sample, DAS, and CAS descriptive statistics and graphs. The output structure mirrors `data/images/DAS/final_run_2026-05-07` and `data/images/CAS/final_run_2026-05-07`, with the final thesis run written under dated `final_run_2026-05-08` folders.

In [1]:

from __future__ import annotations

import json
import os
from datetime import datetime
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

REGISTRIES_DIR = PROJECT_ROOT / "data" / "metadata" / "registries"
SUBREGISTRIES_DIR = REGISTRIES_DIR / "output_subregistries"
IMAGES_DIR = PROJECT_ROOT / "data" / "images"

DOCUMENT_REGISTRY = REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
DAS_CLASSIFICATION_REGISTRY = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"
CAS_CLASSIFICATION_REGISTRY = SUBREGISTRIES_DIR / "CAS_classification_output_registry.csv"

RUN_DATE = "2026-05-08"
RUN_STAMP = datetime.now().strftime("%Y-%m-%d_%H%M%S")

SAMPLE_DIR = IMAGES_DIR / "samples"
DAS_FINAL_DIR = IMAGES_DIR / "DAS" / f"final_run_{RUN_DATE}"
CAS_FINAL_DIR = IMAGES_DIR / "CAS" / f"final_run_{RUN_DATE}"
PREVIOUS_DAS_FINAL_DIR = IMAGES_DIR / "DAS" / "final_run_2026-05-07"
PREVIOUS_CAS_FINAL_DIR = IMAGES_DIR / "CAS" / "final_run_2026-05-07"

for directory in [SAMPLE_DIR, DAS_FINAL_DIR, CAS_FINAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
for root in [DAS_FINAL_DIR, CAS_FINAL_DIR]:
    for scope in ["2024", "2025", "total"]:
        (root / scope).mkdir(parents=True, exist_ok=True)

# Frozen thesis sample years. All total descriptive statistics are scoped to these years.
SAMPLE_YEARS = ["2024", "2025"]

# Last recorded live SCOAP counts used for final thesis sample denominators.
SCOAP_TOTALS = {"2024": 8531, "2025": 8929}
SCOAP_TOTALS["total"] = SCOAP_TOTALS["2024"] + SCOAP_TOTALS["2025"]

PREFERRED_VERSIONS = ["v1.0.5", "v1.0.4", "v1.0.3", "v1.0.2", "v1.0.1", "v1.0.0"]

DAS_ORDER = [
    "no_data", "open_access", "partial_access", "restricted_access",
    "no_access", "incorrect", "unclear",
]
DAS_DISPLAY = {
    "no_data": "No associated data",
    "open_access": "Open access",
    "partial_access": "Partial access",
    "restricted_access": "Restricted access",
    "no_access": "No access",
    "incorrect": "Incorrect / invalid",
    "unclear": "Unclear",
}
DAS_COLORS = {
    "no_data": "#8E8E93", "open_access": "#1D9E75", "partial_access": "#66A61E",
    "restricted_access": "#E6A23C", "no_access": "#D95F02", "incorrect": "#7570B3",
    "unclear": "#6C757D",
}
MCA_ORDER = ["no_code", "restricted_access", "open_access", "unclear", "partial_access", "no_access", "incorrect"]
MCA_COLORS = {
    "no_code": "#888780", "restricted_access": "#D85A30", "open_access": "#1D9E75",
    "unclear": "#7F77DD", "partial_access": "#FAC775", "no_access": "#F09595", "incorrect": "#D4537E",
}
ETA_ORDER = ["no_tool", "proprietary_tool", "open_tool", "mixed_tool", "unknown_status_tool"]
ETA_COLORS = {
    "no_tool": "#888780", "proprietary_tool": "#D85A30", "open_tool": "#1D9E75",
    "mixed_tool": "#7F77DD", "unknown_status_tool": "#FAC775",
}

# Frozen registry snapshot for final descriptive statistics.
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY, dtype=str)
df_base_registry = pd.read_csv(BASE_REGISTRY, dtype=str)
df_raw_registry = pd.read_csv(RAW_REGISTRY, dtype=str)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY, dtype=str)
df_DAS_classification_registry = pd.read_csv(DAS_CLASSIFICATION_REGISTRY, dtype=str)
df_CAS_classification_registry = pd.read_csv(CAS_CLASSIFICATION_REGISTRY, dtype=str)

print(f"Run date: {RUN_DATE}")
print(f"Graph timestamp: {RUN_STAMP}")
print(f"Sample output directory: {SAMPLE_DIR}")
print(f"DAS final output directory: {DAS_FINAL_DIR}")
print(f"CAS final output directory: {CAS_FINAL_DIR}")
print(f"Previous final-run folders mirrored from: {PREVIOUS_DAS_FINAL_DIR} and {PREVIOUS_CAS_FINAL_DIR}")


Run date: 2026-05-08
Graph timestamp: 2026-05-13_123610
Sample output directory: /home/jzupp/projects/medialab_internship/data/images/samples
DAS final output directory: /home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08
CAS final output directory: /home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08
Previous final-run folders mirrored from: /home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-07 and /home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-07


In [ ]:

df_25 = df_document_registry.loc[(df_document_registry["publication_year"]=="2025")]
df_25.shape

(8686, 16)

In [2]:

def bool_true(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().eq("true")


def save_table(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def base_with_docs() -> pd.DataFrame:
    return df_base_registry.merge(
        df_document_registry[["doc_doi", "publication_year", "doc_type", "has_DAS", "has_CAS", "journal"]],
        on="doc_doi",
        how="left",
    )


def resolved_das_classifications() -> pd.DataFrame:
    base_doc = base_with_docs()
    df = (
        df_DAS_classification_registry
        .merge(base_doc, on="output_sha", how="inner")
        .query("output_type == 'classification'")
    )
    df = df[
        df["doc_type"].eq("Scientific study")
        & df["publication_year"].isin(SAMPLE_YEARS)
        & bool_true(df["has_DAS"])
    ].copy()
    df["version_rank"] = df["software_version"].map({v: i for i, v in enumerate(PREFERRED_VERSIONS)}).fillna(999).astype(int)
    df["creation_date_sort"] = pd.to_datetime(df["creation_date"], errors="coerce")
    return (
        df.sort_values(["doc_doi", "version_rank", "creation_date_sort"])
        .drop_duplicates("doc_doi", keep="first")
        .reset_index(drop=True)
    )


def resolved_cas_classifications() -> pd.DataFrame:
    base_doc = base_with_docs()
    df = (
        df_CAS_classification_registry
        .merge(base_doc, on="output_sha", how="inner")
        .query("output_type == 'classification'")
    )
    df = df[
        df["doc_type"].eq("Scientific study")
        & df["publication_year"].isin(SAMPLE_YEARS)
        & bool_true(df["has_CAS"])
    ].copy()
    df["version_rank"] = df["software_version"].map({v: i for i, v in enumerate(PREFERRED_VERSIONS)}).fillna(999).astype(int)
    df["creation_date_sort"] = pd.to_datetime(df["creation_date"], errors="coerce")
    return (
        df.sort_values(["doc_doi", "version_rank", "creation_date_sort"])
        .drop_duplicates("doc_doi", keep="first")
        .reset_index(drop=True)
    )


def count_summary(df: pd.DataFrame, label_col: str, order: list[str]) -> pd.DataFrame:
    counts = df[label_col].value_counts().reindex(order, fill_value=0)
    total = int(counts.sum())
    return pd.DataFrame({
        "label": counts.index,
        "count": counts.values.astype(int),
        "pct": [(value / total * 100) if total else 0 for value in counts.values],
        "n_total": total,
    })


def save_bar(summary: pd.DataFrame, title: str, path: Path, colors: dict[str, str], display_labels: dict[str, str] | None = None) -> Path:
    labels = [display_labels.get(label, label) if display_labels else label for label in summary["label"]]
    fig, ax = plt.subplots(figsize=(9.5, max(4.8, 0.48 * len(labels) + 1.8)))
    ax.barh(labels, summary["count"], color=[colors.get(label, "#5B7DB1") for label in summary["label"]], height=0.65)
    ax.invert_yaxis()
    ax.set_xlabel("Count")
    total = int(summary["n_total"].iloc[0]) if len(summary) else 0
    max_count = max(int(summary["count"].max()), 1) if len(summary) else 1
    ax.set_xlim(0, max_count * 1.22)
    for i, row in enumerate(summary.itertuples(index=False)):
        ax.text(row.count + max_count * 0.02, i, f"{int(row.count):,} ({row.pct:.1f}%)", va="center", fontsize=9)
    ax.set_title(f"{title} (n={total:,})", loc="left", fontsize=13, fontweight="600")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def presence_summary(flag_col: str, year: str | None) -> pd.DataFrame:
    df = df_document_registry[
        df_document_registry["doc_type"].eq("Scientific study")
        & df_document_registry["publication_year"].isin(SAMPLE_YEARS)
        & df_document_registry[flag_col].notna()
    ].copy()
    if year is not None:
        df = df[df["publication_year"].eq(year)].copy()
    short = flag_col[-3:]
    counts = pd.Series({
        f"{short} present": int(bool_true(df[flag_col]).sum()),
        f"No {short}": int((~bool_true(df[flag_col])).sum()),
    })
    total = int(counts.sum())
    return pd.DataFrame({
        "label": counts.index,
        "count": counts.values.astype(int),
        "pct": [(value / total * 100) if total else 0 for value in counts.values],
        "n_total": total,
    })


def save_journal_presence(flag_col: str, year: str | None, out_csv: Path, out_png: Path) -> tuple[Path, Path]:
    df = df_document_registry[
        df_document_registry["doc_type"].eq("Scientific study")
        & df_document_registry["publication_year"].isin(SAMPLE_YEARS)
        & df_document_registry[flag_col].notna()
    ].copy()
    if year is not None:
        df = df[df["publication_year"].eq(year)].copy()
    df["journal"] = df["journal"].fillna("Unknown journal")
    rows = []
    for journal, group in df.groupby("journal"):
        present = int(bool_true(group[flag_col]).sum())
        total = len(group)
        rows.append({
            "journal": journal,
            "present_count": present,
            "absent_count": total - present,
            "total": total,
            "present_pct": present / total * 100 if total else 0,
        })
    summary = pd.DataFrame(rows).sort_values(["total", "present_pct"], ascending=[False, False])
    save_table(summary, out_csv)
    if summary.empty:
        return out_csv, out_png
    fig, ax = plt.subplots(figsize=(10, max(4.8, 0.52 * len(summary) + 1.6)))
    ax.barh(summary["journal"], summary["present_pct"], color="#1D9E75", height=0.65)
    ax.invert_yaxis()
    ax.set_xlim(0, 112)
    ax.set_xlabel("% present")
    for row in summary.itertuples(index=False):
        ax.text(row.present_pct + 1.2, row.journal, f"n={int(row.total):,}", va="center", fontsize=8.5, color="#555")
    short = flag_col[-3:]
    scope = year or "total"
    ax.set_title(f"{short} presence by journal, {scope}", loc="left", fontsize=13, fontweight="600")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_csv, out_png


def save_stacked_by_journal(
    df: pd.DataFrame,
    label_col: str,
    order: list[str],
    colors: dict[str, str],
    title: str,
    out_csv: Path,
    out_png: Path,
    min_segment_pct_label: float | None = None,
) -> tuple[Path, Path]:
    if df.empty:
        save_table(pd.DataFrame(), out_csv)
        return out_csv, out_png
    plot_df = df.copy()
    plot_df["journal"] = plot_df["journal"].fillna("Unknown journal")
    journal_order = plot_df["journal"].value_counts().index.tolist()
    counts = pd.crosstab(plot_df["journal"], plot_df[label_col]).reindex(index=journal_order).reindex(columns=order, fill_value=0)
    totals = counts.sum(axis=1)
    pct = counts.div(totals.replace(0, pd.NA), axis=0).fillna(0) * 100
    counts.to_csv(out_csv)
    fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(counts.index) + 1.7)))
    left = pd.Series(0.0, index=pct.index)
    for label in order:
        widths = pct[label]
        ax.barh(pct.index, widths, left=left, color=colors.get(label, "#5B7DB1"), label=label, height=0.68)
        if min_segment_pct_label is not None:
            for journal, width in widths.items():
                if width >= min_segment_pct_label:
                    ax.text(
                        left[journal] + width / 2,
                        journal,
                        f"{width:.1f}%",
                        va="center",
                        ha="center",
                        fontsize=8,
                        color="white",
                        fontweight="600",
                    )
        left += widths
    for journal, total in totals.items():
        ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
    ax.set_xlim(0, 112)
    ax.invert_yaxis()
    ax.set_title(title, loc="left", fontsize=13, fontweight="600")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=min(4, len(order)), frameon=False)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_csv, out_png


def save_combo_distribution(cas_df: pd.DataFrame, out_csv: Path, out_png: Path, title: str) -> tuple[Path, Path]:
    combo = cas_df.assign(label=lambda item: item["MCA_label"] + " | " + item["ETA_label"])
    counts = combo["label"].value_counts()
    total = int(counts.sum())
    summary = pd.DataFrame({
        "label": counts.index,
        "count": counts.values.astype(int),
        "pct": [(value / total * 100) if total else 0 for value in counts.values],
        "n_total": total,
    })
    save_table(summary, out_csv)
    fig, ax = plt.subplots(figsize=(11, max(4.8, 0.38 * len(summary) + 1.5)))
    ax.barh(summary["label"][::-1], summary["pct"][::-1], color="#5B7DB1", height=0.68)
    for i, row in enumerate(summary.iloc[::-1].itertuples(index=False)):
        ax.text(row.pct + 0.4, i, f"{row.pct:.1f}% ({int(row.count):,})", va="center", fontsize=8.5)
    ax.set_xlim(0, max(float(summary["pct"].max()), 1) + 14 if len(summary) else 1)
    ax.set_xlabel("% of classified CAS")
    ax.set_title(f"{title} (n={total:,})", loc="left", fontsize=13, fontweight="600")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_csv, out_png


## General Sample Descriptives

In [3]:

# General final-sample descriptive statistics and graphs.
year_values = SAMPLE_YEARS
documents = df_document_registry[df_document_registry["publication_year"].isin(year_values)].copy()
raw_base = (
    df_raw_registry
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "publication_year"]], on="doc_doi", how="left")
)

pdf_dois = set(raw_base.loc[raw_base["file_type"].eq("pdf") & raw_base["doc_doi"].notna(), "doc_doi"])
txt_dois = set(raw_base.loc[raw_base["file_type"].eq("txt") & raw_base["doc_doi"].notna(), "doc_doi"])
extraction_output_types = {
    "extracted section", "failed DAS extraction", "failed extraction (missing DAS)",
    "failed CAS extraction", "failed extraction (missing CAS)",
}
marker_dois = set(df_base_registry.loc[df_base_registry["output_type"].isin(extraction_output_types) & df_base_registry["doc_doi"].notna(), "doc_doi"])
function_touched = (
    documents["doc_type"].notna()
    | documents["doc_doi"].isin(marker_dois)
    | documents["has_DAS"].notna()
    | documents["has_CAS"].notna()
)
analysis_eligible = (
    documents["doc_type"].eq("Scientific study")
    & documents["has_DAS"].notna()
    & documents["has_CAS"].notna()
)

sample_rows = []
for scope in ["2024", "2025", "total"]:
    df = documents if scope == "total" else documents[documents["publication_year"].eq(scope)]
    idx = df.index
    live = SCOAP_TOTALS[scope]
    registered = len(df)
    pdf_count = int(df["doc_doi"].isin(pdf_dois).sum())
    txt_count = int(df["doc_doi"].isin(txt_dois).sum())
    extraction_count = int(function_touched.loc[idx].sum())
    analysis_count = int(analysis_eligible.loc[idx].sum())
    non_scientific = int(df["doc_type"].isin(["Erratum", "Editorial expression of concern", "Retraction note"]).sum())
    sample_rows.append({
        "year": scope,
        "scoap_total": live,
        "registered_documents": registered,
        "registered_missing_vs_scoap": live - registered,
        "pdf_registered": pdf_count,
        "pdf_missing_vs_scoap": live - pdf_count,
        "text_converted": txt_count,
        "text_missing_vs_scoap": live - txt_count,
        "reached_extraction_function": extraction_count,
        "extraction_missing_vs_scoap": live - extraction_count,
        "scientific_analysis_sample": analysis_count,
        "analysis_missing_vs_scoap": live - analysis_count,
        "non_scientific_processed": non_scientific,
        "registered_missing_pct": (live - registered) / live * 100,
        "pdf_missing_pct": (live - pdf_count) / live * 100,
        "text_missing_pct": (live - txt_count) / live * 100,
        "extraction_missing_pct": (live - extraction_count) / live * 100,
        "analysis_missing_pct": (live - analysis_count) / live * 100,
    })

sample_summary = pd.DataFrame(sample_rows)
sample_summary_path = save_table(sample_summary, SAMPLE_DIR / f"sample_general_descriptives_{RUN_STAMP}.csv")

doc_type_counts = (
    documents.assign(doc_type=documents["doc_type"].fillna("Not reached / missing doc_type"))
    .groupby(["publication_year", "doc_type"])
    .size()
    .reset_index(name="count")
)
doc_type_counts_path = save_table(doc_type_counts, SAMPLE_DIR / f"sample_doc_type_counts_{RUN_STAMP}.csv")

# Graph 1: processing funnel by year.
funnel_cols = ["registered_documents", "pdf_registered", "text_converted", "reached_extraction_function", "scientific_analysis_sample"]
funnel_labels = ["Registered", "PDF", "Text", "Extraction", "Analysis sample"]
fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(funnel_labels))
width = 0.26
for offset, scope, color in [(-width, "2024", "#4C78A8"), (0, "2025", "#F58518"), (width, "total", "#54A24B")]:
    values = sample_summary.loc[sample_summary["year"].eq(scope), funnel_cols].iloc[0].to_numpy(dtype=float)
    ax.bar(x + offset, values, width=width, label=scope, color=color)
ax.set_xticks(x)
ax.set_xticklabels(funnel_labels, rotation=20, ha="right")
ax.set_ylabel("Records")
ax.set_title("Final sample processing funnel", loc="left", fontsize=13, fontweight="600")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E6E6E6")
ax.set_axisbelow(True)
fig.tight_layout()
funnel_path = SAMPLE_DIR / f"sample_processing_funnel_{RUN_STAMP}.png"
fig.savefig(funnel_path, dpi=180, bbox_inches="tight")
plt.close(fig)

# Graph 2: missing percentage vs SCOAP by stage.
missing_pct_cols = ["registered_missing_pct", "pdf_missing_pct", "text_missing_pct", "extraction_missing_pct", "analysis_missing_pct"]
fig, ax = plt.subplots(figsize=(10.5, 5.2))
x = np.arange(len(funnel_labels))
for scope, color in [("2024", "#4C78A8"), ("2025", "#F58518"), ("total", "#54A24B")]:
    values = sample_summary.loc[sample_summary["year"].eq(scope), missing_pct_cols].iloc[0].to_numpy(dtype=float)
    ax.plot(x, values, marker="o", linewidth=2, label=scope, color=color)
    for xi, value in zip(x, values):
        ax.text(xi, value + 0.05, f"{value:.2f}%", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(funnel_labels, rotation=20, ha="right")
ax.set_ylabel("Missing vs SCOAP (%)")
ax.set_title("Final sample missingness relative to last recorded SCOAP totals", loc="left", fontsize=13, fontweight="600")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E6E6E6")
ax.set_axisbelow(True)
fig.tight_layout()
missing_path = SAMPLE_DIR / f"sample_missing_vs_scoap_{RUN_STAMP}.png"
fig.savefig(missing_path, dpi=180, bbox_inches="tight")
plt.close(fig)

# Graph 3: processed document type composition.
doc_type_total = documents["doc_type"].fillna("Not reached / missing doc_type").value_counts()
fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.barh(doc_type_total.index[::-1], doc_type_total.values[::-1], color="#5B7DB1", height=0.65)
for i, (label, value) in enumerate(zip(doc_type_total.index[::-1], doc_type_total.values[::-1])):
    ax.text(value + max(doc_type_total.max(), 1) * 0.015, i, f"{value:,}", va="center", fontsize=9)
ax.set_xlabel("Records")
ax.set_title("Final processed records by document type, 2024-2025", loc="left", fontsize=13, fontweight="600")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", color="#E6E6E6")
ax.set_axisbelow(True)
fig.tight_layout()
doc_type_plot_path = SAMPLE_DIR / f"sample_doc_type_distribution_{RUN_STAMP}.png"
fig.savefig(doc_type_plot_path, dpi=180, bbox_inches="tight")
plt.close(fig)

print("Saved sample descriptive tables and graphs:")
for path in [sample_summary_path, doc_type_counts_path, funnel_path, missing_path, doc_type_plot_path]:
    print(path)

display(sample_summary)
display(doc_type_counts)


Saved sample descriptive tables and graphs:
/home/jzupp/projects/medialab_internship/data/images/samples/sample_general_descriptives_2026-05-08_014103.csv
/home/jzupp/projects/medialab_internship/data/images/samples/sample_doc_type_counts_2026-05-08_014103.csv
/home/jzupp/projects/medialab_internship/data/images/samples/sample_processing_funnel_2026-05-08_014103.png
/home/jzupp/projects/medialab_internship/data/images/samples/sample_missing_vs_scoap_2026-05-08_014103.png
/home/jzupp/projects/medialab_internship/data/images/samples/sample_doc_type_distribution_2026-05-08_014103.png


,year,scoap_total,registered_documents,registered_missing_vs_scoap,pdf_registered,pdf_missing_vs_scoap,text_converted,text_missing_vs_scoap,reached_extraction_function,extraction_missing_vs_scoap,scientific_analysis_sample,analysis_missing_vs_scoap,non_scientific_processed,registered_missing_pct,pdf_missing_pct,text_missing_pct,extraction_missing_pct,analysis_missing_pct
0,2024,8531,8525,6,8514,17,8511,20,8511,20,8442,89,69,0.070332,0.199273,0.234439,0.234439,1.043254
1,2025,8929,8686,243,8686,243,8674,255,8674,255,8634,295,40,2.721469,2.721469,2.855863,2.855863,3.303841
2,total,17460,17211,249,17200,260,17185,275,17185,275,17076,384,109,1.426117,1.489118,1.575029,1.575029,2.199313


,publication_year,doc_type,count
0,2024,Erratum,69
1,2024,Not reached / missing doc_type,14
2,2024,Scientific study,8442
3,2025,Editorial expression of concern,1
4,2025,Erratum,38
5,2025,Not reached / missing doc_type,12
6,2025,Retraction note,1
7,2025,Scientific study,8634


## DAS Descriptive Statistics

In [3]:

# DAS descriptive statistics and graphs, mirroring data/images/DAS/final_run_2026-05-07.
das_final = resolved_das_classifications()
das_outputs = []
for year in ["2024", "2025", None]:
    scope = year or "total"
    out_dir = DAS_FINAL_DIR / scope
    title_scope = scope

    presence = presence_summary("has_DAS", year)
    presence_csv = save_table(presence, out_dir / f"DAS_presence_{scope}_scientific_studies.csv")
    presence_png = save_bar(
        presence,
        f"DAS presence in {title_scope} scientific studies",
        out_dir / f"DAS_presence_{scope}_scientific_studies.png",
        {"DAS present": "#1D9E75", "No DAS": "#9AA0A6"},
    )
    das_outputs.extend([presence_csv, presence_png])

    das_outputs.extend(save_journal_presence(
        "has_DAS",
        year,
        out_dir / f"DAS_presence_by_journal_{scope}_scientific_studies.csv",
        out_dir / f"DAS_presence_by_journal_{scope}_scientific_studies.png",
    ))

    das_scope = das_final if year is None else das_final[das_final["publication_year"].eq(year)].copy()
    classification = count_summary(das_scope, "label", DAS_ORDER)
    class_csv = save_table(classification, out_dir / f"DAS_classification_{scope}_studies.csv")
    class_png = save_bar(
        classification,
        f"DAS classification output for {title_scope} studies",
        out_dir / f"DAS_classification_{scope}_studies.png",
        DAS_COLORS,
        DAS_DISPLAY,
    )
    das_outputs.extend([class_csv, class_png])

    das_scope_excluding_no_data = das_scope[~das_scope["label"].eq("no_data")].copy()
    classification_excluding_no_data = count_summary(
        das_scope_excluding_no_data,
        "label",
        [label for label in DAS_ORDER if label != "no_data"],
    )
    class_excluding_no_data_csv = save_table(
        classification_excluding_no_data,
        out_dir / f"DAS_classification_excluding_no_data_{scope}_studies.csv",
    )
    class_excluding_no_data_png = save_bar(
        classification_excluding_no_data,
        f"DAS classification output for {title_scope} studies, excluding no-data statements",
        out_dir / f"DAS_classification_excluding_no_data_{scope}_studies.png",
        DAS_COLORS,
        DAS_DISPLAY,
    )
    das_outputs.extend([class_excluding_no_data_csv, class_excluding_no_data_png])

    access_condition_labels = ["open_access", "partial_access", "restricted_access", "no_access"]
    das_scope_access_conditions = das_scope[das_scope["label"].isin(access_condition_labels)].copy()
    access_condition_summary = count_summary(
        das_scope_access_conditions,
        "label",
        access_condition_labels,
    )
    access_condition_csv = save_table(
        access_condition_summary,
        out_dir / f"DAS_classification_access_conditions_only_{scope}_studies.csv",
    )
    access_condition_png = save_bar(
        access_condition_summary,
        f"DAS access-condition classifications for {title_scope} studies",
        out_dir / f"DAS_classification_access_conditions_only_{scope}_studies.png",
        DAS_COLORS,
        DAS_DISPLAY,
    )
    das_outputs.extend([access_condition_csv, access_condition_png])

    das_by_journal_label_threshold = None
    if scope == "total":
        total_counts = pd.crosstab(das_scope["journal"].fillna("Unknown journal"), das_scope["label"]).reindex(columns=DAS_ORDER, fill_value=0)
        total_pct = total_counts.div(total_counts.sum(axis=1).replace(0, pd.NA), axis=0).fillna(0) * 100
        das_by_journal_label_threshold = float(total_pct.loc["Progress of Theoretical and Experimental Physics", "open_access"])

    das_outputs.extend(save_stacked_by_journal(
        das_scope,
        "label",
        DAS_ORDER,
        DAS_COLORS,
        f"DAS classification by journal, {title_scope}",
        out_dir / f"DAS_classification_by_journal_{scope}_studies.csv",
        out_dir / f"DAS_classification_by_journal_{scope}_studies.png",
        min_segment_pct_label=das_by_journal_label_threshold,
    ))

    if scope == "total":
        category_counts = pd.crosstab(
            das_scope["journal"].fillna("Unknown journal"),
            das_scope["label"],
        ).reindex(columns=DAS_ORDER, fill_value=0)
        category_totals = category_counts.sum(axis=1).rename("journal_total")
        category_pct = category_counts.div(category_totals.replace(0, pd.NA), axis=0).fillna(0) * 100
        category_share_summary = pd.DataFrame({"journal": category_counts.index, "journal_total": category_totals.values})
        for label in DAS_ORDER:
            category_share_summary[f"{label}_count"] = category_counts[label].values.astype(int)
            category_share_summary[f"{label}_pct"] = category_pct[label].values
        category_share_summary = category_share_summary.sort_values("journal_total", ascending=False)
        category_share_csv = save_table(
            category_share_summary,
            out_dir / "DAS_classification_category_shares_by_journal_total_studies.csv",
        )

        plot_counts = category_counts.reindex(category_share_summary["journal"])
        plot_pct = category_pct.reindex(category_share_summary["journal"])
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_counts.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct.index)
        for label in DAS_ORDER:
            widths = plot_pct[label]
            ax.barh(
                plot_pct.index,
                widths,
                left=left,
                color=DAS_COLORS.get(label, "#5B7DB1"),
                label=DAS_DISPLAY.get(label, label),
                height=0.68,
            )
            for journal, width in widths.items():
                if width >= das_by_journal_label_threshold:
                    ax.text(
                        left[journal] + width / 2,
                        journal,
                        f"{width:.1f}%",
                        va="center",
                        ha="center",
                        fontsize=8,
                        color="white",
                        fontweight="600",
                    )
            left += widths
        for journal, total in category_totals.reindex(plot_pct.index).items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title("DAS classification category shares by journal, total", loc="left", fontsize=13, fontweight="600")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
        fig.tight_layout()
        category_share_png = out_dir / "DAS_classification_category_shares_by_journal_total_studies.png"
        fig.savefig(category_share_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        das_outputs.extend([category_share_csv, category_share_png])

        category_counts_excluding_no_data = category_counts.drop(columns=["no_data"])
        category_totals_excluding_no_data = category_counts_excluding_no_data.sum(axis=1).rename("journal_total_excluding_no_data")
        category_pct_excluding_no_data = (
            category_counts_excluding_no_data
            .div(category_totals_excluding_no_data.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        category_share_excluding_no_data = pd.DataFrame({
            "journal": category_counts_excluding_no_data.index,
            "journal_total_excluding_no_data": category_totals_excluding_no_data.values,
        })
        for label in [label for label in DAS_ORDER if label != "no_data"]:
            category_share_excluding_no_data[f"{label}_count"] = category_counts_excluding_no_data[label].values.astype(int)
            category_share_excluding_no_data[f"{label}_pct"] = category_pct_excluding_no_data[label].values
        category_share_excluding_no_data = category_share_excluding_no_data.sort_values(
            "journal_total_excluding_no_data",
            ascending=False,
        )
        category_share_excluding_no_data_csv = save_table(
            category_share_excluding_no_data,
            out_dir / "DAS_classification_category_shares_by_journal_excluding_no_data_total_studies.csv",
        )

        plot_counts_excluding_no_data = category_counts_excluding_no_data.reindex(category_share_excluding_no_data["journal"])
        plot_pct_excluding_no_data = category_pct_excluding_no_data.reindex(category_share_excluding_no_data["journal"])
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_counts_excluding_no_data.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct_excluding_no_data.index)
        for label in [label for label in DAS_ORDER if label != "no_data"]:
            widths = plot_pct_excluding_no_data[label]
            ax.barh(
                plot_pct_excluding_no_data.index,
                widths,
                left=left,
                color=DAS_COLORS.get(label, "#5B7DB1"),
                label=DAS_DISPLAY.get(label, label),
                height=0.68,
            )
            for journal, width in widths.items():
                if width >= das_by_journal_label_threshold:
                    ax.text(
                        left[journal] + width / 2,
                        journal,
                        f"{width:.1f}%",
                        va="center",
                        ha="center",
                        fontsize=8,
                        color="white",
                        fontweight="600",
                    )
            left += widths
        for journal, total in category_totals_excluding_no_data.reindex(plot_pct_excluding_no_data.index).items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title(
            "DAS classification category shares by journal, excluding no-data statements",
            loc="left",
            fontsize=13,
            fontweight="600",
        )
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
        fig.tight_layout()
        category_share_excluding_no_data_png = out_dir / "DAS_classification_category_shares_by_journal_excluding_no_data_total_studies.png"
        fig.savefig(category_share_excluding_no_data_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        das_outputs.extend([category_share_excluding_no_data_csv, category_share_excluding_no_data_png])

        category_counts_access_conditions = category_counts[access_condition_labels]
        category_totals_access_conditions = category_counts_access_conditions.sum(axis=1).rename("journal_total_access_conditions_only")
        category_pct_access_conditions = (
            category_counts_access_conditions
            .div(category_totals_access_conditions.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        category_share_access_conditions = pd.DataFrame({
            "journal": category_counts_access_conditions.index,
            "journal_total_access_conditions_only": category_totals_access_conditions.values,
        })
        for label in access_condition_labels:
            category_share_access_conditions[f"{label}_count"] = category_counts_access_conditions[label].values.astype(int)
            category_share_access_conditions[f"{label}_pct"] = category_pct_access_conditions[label].values
        category_share_access_conditions = category_share_access_conditions.sort_values(
            "journal_total_access_conditions_only",
            ascending=False,
        )
        category_share_access_conditions_csv = save_table(
            category_share_access_conditions,
            out_dir / "DAS_classification_category_shares_by_journal_access_conditions_only_total_studies.csv",
        )

        plot_pct_access_conditions = category_pct_access_conditions.reindex(category_share_access_conditions["journal"])
        plot_totals_access_conditions = category_totals_access_conditions.reindex(plot_pct_access_conditions.index)
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_pct_access_conditions.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct_access_conditions.index)
        for label in access_condition_labels:
            widths = plot_pct_access_conditions[label]
            ax.barh(
                plot_pct_access_conditions.index,
                widths,
                left=left,
                color=DAS_COLORS.get(label, "#5B7DB1"),
                label=DAS_DISPLAY.get(label, label),
                height=0.68,
            )
            for journal, width in widths.items():
                if width >= das_by_journal_label_threshold:
                    ax.text(
                        left[journal] + width / 2,
                        journal,
                        f"{width:.1f}%",
                        va="center",
                        ha="center",
                        fontsize=8,
                        color="white",
                        fontweight="600",
                    )
            left += widths
        for journal, total in plot_totals_access_conditions.items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title(
            "DAS access-condition category shares by journal",
            loc="left",
            fontsize=13,
            fontweight="600",
        )
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False)
        fig.tight_layout()
        category_share_access_conditions_png = out_dir / "DAS_classification_category_shares_by_journal_access_conditions_only_total_studies.png"
        fig.savefig(category_share_access_conditions_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        das_outputs.extend([category_share_access_conditions_csv, category_share_access_conditions_png])



# DAS year-over-year presence differences for the frozen thesis sample.
year_diff_dir = DAS_FINAL_DIR / "year_differences"
year_diff_dir.mkdir(parents=True, exist_ok=True)

presence_2024 = presence_summary("has_DAS", "2024")
presence_2025 = presence_summary("has_DAS", "2025")

def presence_row(summary: pd.DataFrame, label: str) -> pd.Series:
    return summary.loc[summary["label"].eq(label)].iloc[0]

das_present_2024 = presence_row(presence_2024, "DAS present")
das_present_2025 = presence_row(presence_2025, "DAS present")
no_das_2024 = presence_row(presence_2024, "No DAS")
no_das_2025 = presence_row(presence_2025, "No DAS")

year_diff_rows = []
for label, row_2024, row_2025 in [
    ("DAS present", das_present_2024, das_present_2025),
    ("No DAS", no_das_2024, no_das_2025),
]:
    pct_2024 = float(row_2024["pct"])
    pct_2025 = float(row_2025["pct"])
    year_diff_rows.append({
        "label": label,
        "count_2024": int(row_2024["count"]),
        "n_total_2024": int(row_2024["n_total"]),
        "pct_2024": pct_2024,
        "count_2025": int(row_2025["count"]),
        "n_total_2025": int(row_2025["n_total"]),
        "pct_2025": pct_2025,
        "count_change_2025_minus_2024": int(row_2025["count"]) - int(row_2024["count"]),
        "percentage_point_change_2025_minus_2024": pct_2025 - pct_2024,
        "relative_percent_change_2025_vs_2024": ((pct_2025 - pct_2024) / pct_2024 * 100) if pct_2024 else pd.NA,
    })

das_presence_year_diff = pd.DataFrame(year_diff_rows)
year_diff_csv = save_table(
    das_presence_year_diff,
    year_diff_dir / "DAS_presence_year_differences_2024_vs_2025.csv",
)

plot_df = das_presence_year_diff.set_index("label")
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(plot_df.index))
width = 0.34
ax.bar(x - width / 2, plot_df["pct_2024"], width, label="2024", color="#9AA0A6")
ax.bar(x + width / 2, plot_df["pct_2025"], width, label="2025", color="#1D9E75")
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("% of scientific studies")
ax.set_ylim(0, 100)
ax.set_title("DAS presence, 2024 vs 2025", loc="left", fontsize=13, fontweight="600")
for i, label in enumerate(plot_df.index):
    for offset, year_col in [(-width / 2, "pct_2024"), (width / 2, "pct_2025")]:
        value = plot_df.loc[label, year_col]
        ax.text(i + offset, value + 1.5, f"{value:.1f}%", ha="center", va="bottom", fontsize=9)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
year_diff_png = year_diff_dir / "DAS_presence_year_differences_2024_vs_2025.png"
fig.savefig(year_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

delta_plot = das_presence_year_diff[["label", "percentage_point_change_2025_minus_2024"]].copy()
fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in delta_plot["percentage_point_change_2025_minus_2024"]]
ax.barh(delta_plot["label"], delta_plot["percentage_point_change_2025_minus_2024"], color=colors, height=0.6)
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_xlim(-70, 70)
ax.set_xlabel("Percentage-point change, 2025 minus 2024")
ax.set_title("DAS presence year difference", loc="left", fontsize=13, fontweight="600")
for row in delta_plot.itertuples(index=False):
    value = row.percentage_point_change_2025_minus_2024
    ax.text(value / 2, row.label, f"{value:+.2f} pp", va="center", ha="center", fontsize=9, color="white", fontweight="600")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=8)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
year_diff_delta_png = year_diff_dir / "DAS_presence_percentage_point_change_2024_vs_2025.png"
fig.savefig(year_diff_delta_png, dpi=180, bbox_inches="tight")
plt.close(fig)

# DAS presence percentage-point differences by journal.
journal_2024 = pd.read_csv(DAS_FINAL_DIR / "2024" / "DAS_presence_by_journal_2024_scientific_studies.csv")
journal_2025 = pd.read_csv(DAS_FINAL_DIR / "2025" / "DAS_presence_by_journal_2025_scientific_studies.csv")
journal_diff = journal_2024.merge(
    journal_2025,
    on="journal",
    how="outer",
    suffixes=("_2024", "_2025"),
).fillna(0)
for col in ["present_count_2024", "absent_count_2024", "total_2024", "present_count_2025", "absent_count_2025", "total_2025"]:
    journal_diff[col] = journal_diff[col].astype(int)
journal_diff["present_pct_2024"] = np.where(
    journal_diff["total_2024"].gt(0),
    journal_diff["present_pct_2024"].astype(float),
    np.nan,
)
journal_diff["present_pct_2025"] = np.where(
    journal_diff["total_2025"].gt(0),
    journal_diff["present_pct_2025"].astype(float),
    np.nan,
)
journal_diff["percentage_point_change_2025_minus_2024"] = journal_diff["present_pct_2025"] - journal_diff["present_pct_2024"]
journal_diff["relative_percent_change_2025_vs_2024"] = np.where(
    journal_diff["present_pct_2024"].ne(0) & journal_diff["present_pct_2024"].notna() & journal_diff["present_pct_2025"].notna(),
    journal_diff["percentage_point_change_2025_minus_2024"] / journal_diff["present_pct_2024"] * 100,
    np.nan,
)
journal_diff["total_2024_2025"] = journal_diff["total_2024"] + journal_diff["total_2025"]
journal_diff = journal_diff.sort_values("percentage_point_change_2025_minus_2024", ascending=False, na_position="last")
journal_diff_csv = save_table(
    journal_diff,
    year_diff_dir / "DAS_presence_by_journal_percentage_point_change_2024_vs_2025.csv",
)

fig, ax = plt.subplots(figsize=(11, max(5.2, 0.55 * len(journal_diff) + 1.8)))
plot_journal = journal_diff.dropna(subset=["percentage_point_change_2025_minus_2024"]).sort_values("percentage_point_change_2025_minus_2024")
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in plot_journal["percentage_point_change_2025_minus_2024"]]
ax.barh(plot_journal["journal"], plot_journal["percentage_point_change_2025_minus_2024"], color=colors, height=0.65)
ax.axvline(0, color="#333333", linewidth=0.9)
max_abs = max(abs(plot_journal["percentage_point_change_2025_minus_2024"]).max(), 1)
ax.set_xlim(-max_abs * 1.25, max_abs * 1.25)
ax.set_xlabel("Percentage-point change in DAS presence, 2025 minus 2024")
ax.set_title("DAS presence by journal: percentage-point change", loc="left", fontsize=13, fontweight="600")
for row in plot_journal.itertuples(index=False):
    value = row.percentage_point_change_2025_minus_2024
    ha = "left" if value >= 0 else "right"
    offset = max_abs * 0.035 if value >= 0 else -max_abs * 0.035
    ax.text(value + offset, row.journal, f"{value:+.2f} pp", va="center", ha=ha, fontsize=8.5)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
journal_diff_png = year_diff_dir / "DAS_presence_by_journal_percentage_point_change_2024_vs_2025.png"
fig.savefig(journal_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

das_outputs.extend([year_diff_csv, year_diff_png, year_diff_delta_png, journal_diff_csv, journal_diff_png])



# DAS classification category-share percentage-point differences.
classification_2024 = pd.read_csv(DAS_FINAL_DIR / "2024" / "DAS_classification_2024_studies.csv")
classification_2025 = pd.read_csv(DAS_FINAL_DIR / "2025" / "DAS_classification_2025_studies.csv")
classification_diff = classification_2024.merge(
    classification_2025,
    on="label",
    how="outer",
    suffixes=("_2024", "_2025"),
).fillna(0)
classification_diff["label"] = pd.Categorical(classification_diff["label"], categories=DAS_ORDER, ordered=True)
classification_diff = classification_diff.sort_values("label")
for col in ["count_2024", "n_total_2024", "count_2025", "n_total_2025"]:
    classification_diff[col] = classification_diff[col].astype(int)
classification_diff["pct_2024"] = classification_diff["pct_2024"].astype(float)
classification_diff["pct_2025"] = classification_diff["pct_2025"].astype(float)
classification_diff["count_change_2025_minus_2024"] = classification_diff["count_2025"] - classification_diff["count_2024"]
classification_diff["percentage_point_change_2025_minus_2024"] = classification_diff["pct_2025"] - classification_diff["pct_2024"]
classification_diff["relative_percent_change_2025_vs_2024"] = np.where(
    classification_diff["pct_2024"].ne(0),
    classification_diff["percentage_point_change_2025_minus_2024"] / classification_diff["pct_2024"] * 100,
    np.nan,
)
classification_diff_csv = save_table(
    classification_diff,
    year_diff_dir / "DAS_classification_percentage_point_change_2024_vs_2025.csv",
)

plot_classification_diff = classification_diff.sort_values("percentage_point_change_2025_minus_2024")
fig, ax = plt.subplots(figsize=(9.5, max(4.8, 0.55 * len(plot_classification_diff) + 1.8)))
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in plot_classification_diff["percentage_point_change_2025_minus_2024"]]
labels = [DAS_DISPLAY.get(str(label), str(label)) for label in plot_classification_diff["label"]]
ax.barh(labels, plot_classification_diff["percentage_point_change_2025_minus_2024"], color=colors, height=0.65)
ax.axvline(0, color="#333333", linewidth=0.9)
max_abs = max(abs(plot_classification_diff["percentage_point_change_2025_minus_2024"]).max(), 1)
ax.set_xlim(-max_abs * 1.35, max_abs * 1.35)
ax.set_xlabel("Percentage-point change in category share, 2025 minus 2024")
ax.set_title("DAS classification category-share changes", loc="left", fontsize=13, fontweight="600")
for label, value in zip(labels, plot_classification_diff["percentage_point_change_2025_minus_2024"]):
    ha = "left" if value >= 0 else "right"
    offset = max_abs * 0.045 if value >= 0 else -max_abs * 0.045
    ax.text(value + offset, label, f"{value:+.2f} pp", va="center", ha=ha, fontsize=9)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
classification_diff_png = year_diff_dir / "DAS_classification_percentage_point_change_2024_vs_2025.png"
fig.savefig(classification_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

das_outputs.extend([classification_diff_csv, classification_diff_png])

# DAS access-condition-only category-share percentage-point differences.
access_condition_labels = ["open_access", "partial_access", "restricted_access", "no_access"]
access_conditions_2024 = pd.read_csv(DAS_FINAL_DIR / "2024" / "DAS_classification_access_conditions_only_2024_studies.csv")
access_conditions_2025 = pd.read_csv(DAS_FINAL_DIR / "2025" / "DAS_classification_access_conditions_only_2025_studies.csv")
access_condition_diff = access_conditions_2024.merge(
    access_conditions_2025,
    on="label",
    how="outer",
    suffixes=("_2024", "_2025"),
).fillna(0)
access_condition_diff["label"] = pd.Categorical(access_condition_diff["label"], categories=access_condition_labels, ordered=True)
access_condition_diff = access_condition_diff.sort_values("label")
for col in ["count_2024", "n_total_2024", "count_2025", "n_total_2025"]:
    access_condition_diff[col] = access_condition_diff[col].astype(int)
access_condition_diff["pct_2024"] = access_condition_diff["pct_2024"].astype(float)
access_condition_diff["pct_2025"] = access_condition_diff["pct_2025"].astype(float)
access_condition_diff["count_change_2025_minus_2024"] = access_condition_diff["count_2025"] - access_condition_diff["count_2024"]
access_condition_diff["percentage_point_change_2025_minus_2024"] = access_condition_diff["pct_2025"] - access_condition_diff["pct_2024"]
access_condition_diff["relative_percent_change_2025_vs_2024"] = np.where(
    access_condition_diff["pct_2024"].ne(0),
    access_condition_diff["percentage_point_change_2025_minus_2024"] / access_condition_diff["pct_2024"] * 100,
    np.nan,
)
access_condition_diff_csv = save_table(
    access_condition_diff,
    year_diff_dir / "DAS_access_condition_category_share_percentage_point_change_2024_vs_2025.csv",
)

plot_access_condition_diff = access_condition_diff.sort_values("percentage_point_change_2025_minus_2024")
fig, ax = plt.subplots(figsize=(9.5, max(4.8, 0.55 * len(plot_access_condition_diff) + 1.8)))
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in plot_access_condition_diff["percentage_point_change_2025_minus_2024"]]
labels = [DAS_DISPLAY.get(str(label), str(label)) for label in plot_access_condition_diff["label"]]
ax.barh(labels, plot_access_condition_diff["percentage_point_change_2025_minus_2024"], color=colors, height=0.65)
ax.axvline(0, color="#333333", linewidth=0.9)
max_abs = max(abs(plot_access_condition_diff["percentage_point_change_2025_minus_2024"]).max(), 1)
ax.set_xlim(-max_abs * 1.35, max_abs * 1.35)
ax.set_xlabel("Percentage-point change in access-condition share, 2025 minus 2024")
ax.set_title("DAS access-condition category-share changes", loc="left", fontsize=13, fontweight="600")
for label, value in zip(labels, plot_access_condition_diff["percentage_point_change_2025_minus_2024"]):
    ha = "left" if value >= 0 else "right"
    offset = max_abs * 0.045 if value >= 0 else -max_abs * 0.045
    ax.text(value + offset, label, f"{value:+.2f} pp", va="center", ha=ha, fontsize=9)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
access_condition_diff_png = year_diff_dir / "DAS_access_condition_category_share_percentage_point_change_2024_vs_2025.png"
fig.savefig(access_condition_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

das_outputs.extend([access_condition_diff_csv, access_condition_diff_png])

print(f"Resolved DAS classification rows: {len(das_final):,}")
print(f"Unique DAS-classified publications: {das_final['doc_doi'].nunique():,}")
print("Saved DAS outputs:")
for path in das_outputs:
    print(path)

display(das_final["publication_year"].value_counts().sort_index().rename("DAS classified records"))
display(count_summary(das_final, "label", DAS_ORDER))


Resolved DAS classification rows: 10,843
Unique DAS-classified publications: 10,843
Saved DAS outputs:
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_presence_2024_scientific_studies.csv
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_presence_2024_scientific_studies.png
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_presence_by_journal_2024_scientific_studies.csv
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_presence_by_journal_2024_scientific_studies.png
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_classification_2024_studies.csv
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_classification_2024_studies.png
/home/jzupp/projects/medialab_internship/data/images/DAS/final_run_2026-05-08/2024/DAS_classification_excluding_no_data_2024_studies.csv
/hom

publication_year
2024    2964
2025    7879
Name: DAS classified records, dtype: int64

,label,count,pct,n_total
0,no_data,7237,66.743521,10843
1,open_access,1228,11.325279,10843
2,partial_access,80,0.737803,10843
3,restricted_access,1976,18.223739,10843
4,no_access,35,0.322789,10843
5,incorrect,32,0.295121,10843
6,unclear,255,2.351748,10843


## CAS Descriptive Statistics

In [3]:

# CAS descriptive statistics and graphs, mirroring data/images/CAS/final_run_2026-05-07.
cas_final = resolved_cas_classifications()
cas_outputs = []
for year in ["2024", "2025", None]:
    scope = year or "total"
    out_dir = CAS_FINAL_DIR / scope
    title_scope = scope

    presence = presence_summary("has_CAS", year)
    presence_csv = save_table(presence, out_dir / f"CAS_presence_{scope}_scientific_studies.csv")
    presence_png = save_bar(
        presence,
        f"CAS presence in {title_scope} scientific studies",
        out_dir / f"CAS_presence_{scope}_scientific_studies.png",
        {"CAS present": "#1D9E75", "No CAS": "#9AA0A6"},
    )
    cas_outputs.extend([presence_csv, presence_png])

    cas_outputs.extend(save_journal_presence(
        "has_CAS",
        year,
        out_dir / f"CAS_presence_by_journal_{scope}_scientific_studies.csv",
        out_dir / f"CAS_presence_by_journal_{scope}_scientific_studies.png",
    ))

    cas_scope = cas_final if year is None else cas_final[cas_final["publication_year"].eq(year)].copy()

    mca = count_summary(cas_scope, "MCA_label", MCA_ORDER)
    mca_csv = save_table(mca, out_dir / f"MCA_classification_{scope}.csv")
    mca_png = save_bar(
        mca,
        f"CAS MCA classification for {title_scope}",
        out_dir / f"MCA_classification_{scope}.png",
        MCA_COLORS,
    )
    cas_outputs.extend([mca_csv, mca_png])

    cas_scope_excluding_no_code = cas_scope[~cas_scope["MCA_label"].eq("no_code")].copy()
    mca_excluding_no_code = count_summary(
        cas_scope_excluding_no_code,
        "MCA_label",
        [label for label in MCA_ORDER if label != "no_code"],
    )
    mca_excluding_no_code_csv = save_table(
        mca_excluding_no_code,
        out_dir / f"MCA_classification_excluding_no_code_{scope}.csv",
    )
    mca_excluding_no_code_png = save_bar(
        mca_excluding_no_code,
        f"CAS MCA classification for {title_scope}, excluding no-code statements",
        out_dir / f"MCA_classification_excluding_no_code_{scope}.png",
        MCA_COLORS,
    )
    cas_outputs.extend([mca_excluding_no_code_csv, mca_excluding_no_code_png])

    mca_access_condition_labels = ["open_access", "partial_access", "restricted_access", "no_access"]
    cas_scope_mca_access_conditions = cas_scope[cas_scope["MCA_label"].isin(mca_access_condition_labels)].copy()
    mca_access_conditions = count_summary(
        cas_scope_mca_access_conditions,
        "MCA_label",
        mca_access_condition_labels,
    )
    mca_access_conditions_csv = save_table(
        mca_access_conditions,
        out_dir / f"MCA_classification_access_conditions_only_{scope}.csv",
    )
    mca_access_conditions_png = save_bar(
        mca_access_conditions,
        f"CAS MCA access-condition classifications for {title_scope}",
        out_dir / f"MCA_classification_access_conditions_only_{scope}.png",
        MCA_COLORS,
    )
    cas_outputs.extend([mca_access_conditions_csv, mca_access_conditions_png])

    cas_scope_with_est = cas_scope[~cas_scope["ETA_label"].eq("no_tool")].copy()
    mca_when_est_mentioned = count_summary(
        cas_scope_with_est,
        "MCA_label",
        MCA_ORDER,
    )
    mca_when_est_mentioned_csv = save_table(
        mca_when_est_mentioned,
        out_dir / f"MCA_classification_when_EST_mentioned_{scope}.csv",
    )
    mca_when_est_mentioned_png = save_bar(
        mca_when_est_mentioned,
        f"CAS MCA classification for {title_scope}, when EST is mentioned",
        out_dir / f"MCA_classification_when_EST_mentioned_{scope}.png",
        MCA_COLORS,
    )
    cas_outputs.extend([mca_when_est_mentioned_csv, mca_when_est_mentioned_png])

    eta = count_summary(cas_scope, "ETA_label", ETA_ORDER)
    eta_csv = save_table(eta, out_dir / f"ETA_classification_{scope}.csv")
    eta_png = save_bar(
        eta,
        f"CAS ETA classification for {title_scope}",
        out_dir / f"ETA_classification_{scope}.png",
        ETA_COLORS,
    )
    cas_outputs.extend([eta_csv, eta_png])

    cas_scope_excluding_no_tool = cas_scope[~cas_scope["ETA_label"].eq("no_tool")].copy()
    eta_excluding_no_tool = count_summary(
        cas_scope_excluding_no_tool,
        "ETA_label",
        [label for label in ETA_ORDER if label != "no_tool"],
    )
    eta_excluding_no_tool_csv = save_table(
        eta_excluding_no_tool,
        out_dir / f"ETA_classification_excluding_no_tool_{scope}.csv",
    )
    eta_excluding_no_tool_png = save_bar(
        eta_excluding_no_tool,
        f"CAS ETA classification for {title_scope}, excluding no-tool statements",
        out_dir / f"ETA_classification_excluding_no_tool_{scope}.png",
        ETA_COLORS,
    )
    cas_outputs.extend([eta_excluding_no_tool_csv, eta_excluding_no_tool_png])

    eta_access_condition_labels = ["proprietary_tool", "open_tool", "mixed_tool"]
    cas_scope_eta_access_conditions = cas_scope[cas_scope["ETA_label"].isin(eta_access_condition_labels)].copy()
    eta_access_conditions = count_summary(
        cas_scope_eta_access_conditions,
        "ETA_label",
        eta_access_condition_labels,
    )
    eta_access_conditions_csv = save_table(
        eta_access_conditions,
        out_dir / f"ETA_classification_access_conditions_only_{scope}.csv",
    )
    eta_access_conditions_png = save_bar(
        eta_access_conditions,
        f"CAS ETA access-condition classifications for {title_scope}",
        out_dir / f"ETA_classification_access_conditions_only_{scope}.png",
        ETA_COLORS,
    )
    cas_outputs.extend([eta_access_conditions_csv, eta_access_conditions_png])

    cas_outputs.extend(save_combo_distribution(
        cas_scope,
        out_dir / f"CAS_combined_MCA_ETA_distribution_{scope}.csv",
        out_dir / f"CAS_combined_MCA_ETA_distribution_{scope}.png",
        f"CAS combined MCA/ETA distribution for {title_scope}",
    ))

    cas_scope_combo_excluding_no_code_no_tool = cas_scope[
        ~cas_scope["MCA_label"].eq("no_code")
        & ~cas_scope["ETA_label"].eq("no_tool")
    ].copy()
    cas_outputs.extend(save_combo_distribution(
        cas_scope_combo_excluding_no_code_no_tool,
        out_dir / f"CAS_combined_MCA_ETA_distribution_excluding_no_code_no_tool_{scope}.csv",
        out_dir / f"CAS_combined_MCA_ETA_distribution_excluding_no_code_no_tool_{scope}.png",
        f"CAS combined MCA/ETA distribution for {title_scope}, excluding no-code/no-tool combinations",
    ))

    cas_outputs.extend(save_stacked_by_journal(
        cas_scope,
        "MCA_label",
        MCA_ORDER,
        MCA_COLORS,
        f"CAS MCA classification by journal, {title_scope}",
        out_dir / f"MCA_classification_by_journal_{scope}.csv",
        out_dir / f"MCA_classification_by_journal_{scope}.png",
    ))

    if scope == "total":
        mca_counts_by_journal = pd.crosstab(
            cas_scope["journal"].fillna("Unknown journal"),
            cas_scope["MCA_label"],
        ).reindex(columns=MCA_ORDER, fill_value=0)
        mca_totals_by_journal = mca_counts_by_journal.sum(axis=1).rename("journal_total")
        mca_pct_by_journal = (
            mca_counts_by_journal
            .div(mca_totals_by_journal.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        mca_share_summary = pd.DataFrame({
            "journal": mca_counts_by_journal.index,
            "journal_total": mca_totals_by_journal.values,
        })
        for label in MCA_ORDER:
            mca_share_summary[f"{label}_count"] = mca_counts_by_journal[label].values.astype(int)
            mca_share_summary[f"{label}_pct"] = mca_pct_by_journal[label].values
        mca_share_summary = mca_share_summary.sort_values("journal_total", ascending=False)
        mca_share_csv = save_table(
            mca_share_summary,
            out_dir / "MCA_classification_category_shares_by_journal_total.csv",
        )

        mca_counts_excluding_no_code = mca_counts_by_journal.drop(columns=["no_code"])
        mca_totals_excluding_no_code = mca_counts_excluding_no_code.sum(axis=1).rename("journal_total_excluding_no_code")
        mca_pct_excluding_no_code = (
            mca_counts_excluding_no_code
            .div(mca_totals_excluding_no_code.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        mca_share_excluding_no_code = pd.DataFrame({
            "journal": mca_counts_excluding_no_code.index,
            "journal_total_excluding_no_code": mca_totals_excluding_no_code.values,
        })
        for label in [label for label in MCA_ORDER if label != "no_code"]:
            mca_share_excluding_no_code[f"{label}_count"] = mca_counts_excluding_no_code[label].values.astype(int)
            mca_share_excluding_no_code[f"{label}_pct"] = mca_pct_excluding_no_code[label].values
        mca_share_excluding_no_code = mca_share_excluding_no_code.sort_values(
            "journal_total_excluding_no_code",
            ascending=False,
        )
        mca_share_excluding_no_code_csv = save_table(
            mca_share_excluding_no_code,
            out_dir / "MCA_classification_category_shares_by_journal_excluding_no_code_total.csv",
        )

        plot_pct = mca_pct_excluding_no_code.reindex(mca_share_excluding_no_code["journal"])
        plot_totals = mca_totals_excluding_no_code.reindex(plot_pct.index)
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_pct.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct.index)
        for label in [label for label in MCA_ORDER if label != "no_code"]:
            widths = plot_pct[label]
            ax.barh(
                plot_pct.index,
                widths,
                left=left,
                color=MCA_COLORS.get(label, "#5B7DB1"),
                label=label,
                height=0.68,
            )
            for journal, width in widths.items():
                if width >= 7.0:
                    ax.text(
                        left[journal] + width / 2,
                        journal,
                        f"{width:.1f}%",
                        va="center",
                        ha="center",
                        fontsize=8,
                        color="white",
                        fontweight="600",
                    )
            left += widths
        for journal, total in plot_totals.items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title(
            "CAS MCA category shares by journal, excluding no-code statements",
            loc="left",
            fontsize=13,
            fontweight="600",
        )
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
        fig.tight_layout()
        mca_share_excluding_no_code_png = out_dir / "MCA_classification_category_shares_by_journal_excluding_no_code_total.png"
        fig.savefig(mca_share_excluding_no_code_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        cas_outputs.extend([mca_share_csv, mca_share_excluding_no_code_csv, mca_share_excluding_no_code_png])

        mca_counts_access_conditions = mca_counts_by_journal[mca_access_condition_labels]
        mca_totals_access_conditions = mca_counts_access_conditions.sum(axis=1).rename("journal_total_access_conditions_only")
        mca_pct_access_conditions = (
            mca_counts_access_conditions
            .div(mca_totals_access_conditions.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        mca_share_access_conditions = pd.DataFrame({
            "journal": mca_counts_access_conditions.index,
            "journal_total_access_conditions_only": mca_totals_access_conditions.values,
        })
        for label in mca_access_condition_labels:
            mca_share_access_conditions[f"{label}_count"] = mca_counts_access_conditions[label].values.astype(int)
            mca_share_access_conditions[f"{label}_pct"] = mca_pct_access_conditions[label].values
        mca_share_access_conditions = mca_share_access_conditions.sort_values(
            "journal_total_access_conditions_only",
            ascending=False,
        )
        mca_share_access_conditions_csv = save_table(
            mca_share_access_conditions,
            out_dir / "MCA_classification_category_shares_by_journal_access_conditions_only_total.csv",
        )

        plot_pct = mca_pct_access_conditions.reindex(mca_share_access_conditions["journal"])
        plot_totals = mca_totals_access_conditions.reindex(plot_pct.index)
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_pct.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct.index)
        for label in mca_access_condition_labels:
            widths = plot_pct[label]
            ax.barh(plot_pct.index, widths, left=left, color=MCA_COLORS.get(label, "#5B7DB1"), label=label, height=0.68)
            for journal, width in widths.items():
                if width >= 7.0:
                    ax.text(left[journal] + width / 2, journal, f"{width:.1f}%", va="center", ha="center", fontsize=8, color="white", fontweight="600")
            left += widths
        for journal, total in plot_totals.items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title("CAS MCA access-condition category shares by journal", loc="left", fontsize=13, fontweight="600")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=4, frameon=False)
        fig.tight_layout()
        mca_share_access_conditions_png = out_dir / "MCA_classification_category_shares_by_journal_access_conditions_only_total.png"
        fig.savefig(mca_share_access_conditions_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        cas_outputs.extend([mca_share_access_conditions_csv, mca_share_access_conditions_png])
    cas_outputs.extend(save_stacked_by_journal(
        cas_scope,
        "ETA_label",
        ETA_ORDER,
        ETA_COLORS,
        f"CAS ETA classification by journal, {title_scope}",
        out_dir / f"ETA_classification_by_journal_{scope}.csv",
        out_dir / f"ETA_classification_by_journal_{scope}.png",
    ))

    if scope == "total":
        eta_counts_by_journal = pd.crosstab(
            cas_scope["journal"].fillna("Unknown journal"),
            cas_scope["ETA_label"],
        ).reindex(columns=ETA_ORDER, fill_value=0)
        eta_counts_access_conditions = eta_counts_by_journal[eta_access_condition_labels]
        eta_totals_access_conditions = eta_counts_access_conditions.sum(axis=1).rename("journal_total_access_conditions_only")
        eta_pct_access_conditions = (
            eta_counts_access_conditions
            .div(eta_totals_access_conditions.replace(0, pd.NA), axis=0)
            .fillna(0)
            * 100
        )
        eta_share_access_conditions = pd.DataFrame({
            "journal": eta_counts_access_conditions.index,
            "journal_total_access_conditions_only": eta_totals_access_conditions.values,
        })
        for label in eta_access_condition_labels:
            eta_share_access_conditions[f"{label}_count"] = eta_counts_access_conditions[label].values.astype(int)
            eta_share_access_conditions[f"{label}_pct"] = eta_pct_access_conditions[label].values
        eta_share_access_conditions = eta_share_access_conditions.sort_values(
            "journal_total_access_conditions_only",
            ascending=False,
        )
        eta_share_access_conditions_csv = save_table(
            eta_share_access_conditions,
            out_dir / "ETA_classification_category_shares_by_journal_access_conditions_only_total.csv",
        )

        plot_pct = eta_pct_access_conditions.reindex(eta_share_access_conditions["journal"])
        plot_totals = eta_totals_access_conditions.reindex(plot_pct.index)
        fig, ax = plt.subplots(figsize=(11.5, max(4.8, 0.58 * len(plot_pct.index) + 1.7)))
        left = pd.Series(0.0, index=plot_pct.index)
        for label in eta_access_condition_labels:
            widths = plot_pct[label]
            ax.barh(plot_pct.index, widths, left=left, color=ETA_COLORS.get(label, "#5B7DB1"), label=label, height=0.68)
            for journal, width in widths.items():
                if width >= 7.0:
                    ax.text(left[journal] + width / 2, journal, f"{width:.1f}%", va="center", ha="center", fontsize=8, color="white", fontweight="600")
            left += widths
        for journal, total in plot_totals.items():
            ax.text(101.2, journal, f"n={int(total):,}", va="center", fontsize=8.5, color="#555")
        ax.set_xlim(0, 112)
        ax.invert_yaxis()
        ax.set_title("CAS ETA access-condition category shares by journal", loc="left", fontsize=13, fontweight="600")
        ax.spines[["top", "right", "left"]].set_visible(False)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
        fig.tight_layout()
        eta_share_access_conditions_png = out_dir / "ETA_classification_category_shares_by_journal_access_conditions_only_total.png"
        fig.savefig(eta_share_access_conditions_png, dpi=180, bbox_inches="tight")
        plt.close(fig)
        cas_outputs.extend([eta_share_access_conditions_csv, eta_share_access_conditions_png])



# CAS year-over-year differences for the frozen thesis sample.
cas_year_diff_dir = CAS_FINAL_DIR / "year_differences"
cas_year_diff_dir.mkdir(parents=True, exist_ok=True)

cas_presence_2024 = presence_summary("has_CAS", "2024")
cas_presence_2025 = presence_summary("has_CAS", "2025")

def cas_presence_row(summary: pd.DataFrame, label: str) -> pd.Series:
    return summary.loc[summary["label"].eq(label)].iloc[0]

cas_present_2024 = cas_presence_row(cas_presence_2024, "CAS present")
cas_present_2025 = cas_presence_row(cas_presence_2025, "CAS present")
no_cas_2024 = cas_presence_row(cas_presence_2024, "No CAS")
no_cas_2025 = cas_presence_row(cas_presence_2025, "No CAS")

cas_presence_diff_rows = []
for label, row_2024, row_2025 in [
    ("CAS present", cas_present_2024, cas_present_2025),
    ("No CAS", no_cas_2024, no_cas_2025),
]:
    pct_2024 = float(row_2024["pct"])
    pct_2025 = float(row_2025["pct"])
    cas_presence_diff_rows.append({
        "label": label,
        "count_2024": int(row_2024["count"]),
        "n_total_2024": int(row_2024["n_total"]),
        "pct_2024": pct_2024,
        "count_2025": int(row_2025["count"]),
        "n_total_2025": int(row_2025["n_total"]),
        "pct_2025": pct_2025,
        "count_change_2025_minus_2024": int(row_2025["count"]) - int(row_2024["count"]),
        "percentage_point_change_2025_minus_2024": pct_2025 - pct_2024,
        "relative_percent_change_2025_vs_2024": ((pct_2025 - pct_2024) / pct_2024 * 100) if pct_2024 else pd.NA,
    })

cas_presence_year_diff = pd.DataFrame(cas_presence_diff_rows)
cas_presence_year_diff_csv = save_table(
    cas_presence_year_diff,
    cas_year_diff_dir / "CAS_presence_year_differences_2024_vs_2025.csv",
)

plot_df = cas_presence_year_diff.set_index("label")
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(plot_df.index))
width = 0.34
ax.bar(x - width / 2, plot_df["pct_2024"], width, label="2024", color="#9AA0A6")
ax.bar(x + width / 2, plot_df["pct_2025"], width, label="2025", color="#1D9E75")
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("% of scientific studies")
ax.set_ylim(0, 100)
ax.set_title("CAS presence, 2024 vs 2025", loc="left", fontsize=13, fontweight="600")
for i, label in enumerate(plot_df.index):
    for offset, year_col in [(-width / 2, "pct_2024"), (width / 2, "pct_2025")]:
        value = plot_df.loc[label, year_col]
        ax.text(i + offset, value + 1.5, f"{value:.1f}%", ha="center", va="bottom", fontsize=9)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
cas_presence_year_diff_png = cas_year_diff_dir / "CAS_presence_year_differences_2024_vs_2025.png"
fig.savefig(cas_presence_year_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

cas_delta_plot = cas_presence_year_diff[["label", "percentage_point_change_2025_minus_2024"]].copy()
fig, ax = plt.subplots(figsize=(8.5, 4.2))
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in cas_delta_plot["percentage_point_change_2025_minus_2024"]]
ax.barh(cas_delta_plot["label"], cas_delta_plot["percentage_point_change_2025_minus_2024"], color=colors, height=0.6)
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_xlim(-40, 40)
ax.set_xlabel("Percentage-point change, 2025 minus 2024")
ax.set_title("CAS presence year difference", loc="left", fontsize=13, fontweight="600")
for row in cas_delta_plot.itertuples(index=False):
    value = row.percentage_point_change_2025_minus_2024
    ax.text(value / 2, row.label, f"{value:+.2f} pp", va="center", ha="center", fontsize=9, color="white", fontweight="600")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0, pad=8)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
cas_presence_delta_png = cas_year_diff_dir / "CAS_presence_percentage_point_change_2024_vs_2025.png"
fig.savefig(cas_presence_delta_png, dpi=180, bbox_inches="tight")
plt.close(fig)

cas_journal_2024 = pd.read_csv(CAS_FINAL_DIR / "2024" / "CAS_presence_by_journal_2024_scientific_studies.csv")
cas_journal_2025 = pd.read_csv(CAS_FINAL_DIR / "2025" / "CAS_presence_by_journal_2025_scientific_studies.csv")
cas_journal_diff = cas_journal_2024.merge(cas_journal_2025, on="journal", how="outer", suffixes=("_2024", "_2025")).fillna(0)
for col in ["present_count_2024", "absent_count_2024", "total_2024", "present_count_2025", "absent_count_2025", "total_2025"]:
    cas_journal_diff[col] = cas_journal_diff[col].astype(int)
cas_journal_diff["present_pct_2024"] = np.where(cas_journal_diff["total_2024"].gt(0), cas_journal_diff["present_pct_2024"].astype(float), np.nan)
cas_journal_diff["present_pct_2025"] = np.where(cas_journal_diff["total_2025"].gt(0), cas_journal_diff["present_pct_2025"].astype(float), np.nan)
cas_journal_diff["percentage_point_change_2025_minus_2024"] = cas_journal_diff["present_pct_2025"] - cas_journal_diff["present_pct_2024"]
cas_journal_diff["relative_percent_change_2025_vs_2024"] = np.where(
    cas_journal_diff["present_pct_2024"].ne(0) & cas_journal_diff["present_pct_2024"].notna() & cas_journal_diff["present_pct_2025"].notna(),
    cas_journal_diff["percentage_point_change_2025_minus_2024"] / cas_journal_diff["present_pct_2024"] * 100,
    np.nan,
)
cas_journal_diff["total_2024_2025"] = cas_journal_diff["total_2024"] + cas_journal_diff["total_2025"]
cas_journal_diff = cas_journal_diff.sort_values("percentage_point_change_2025_minus_2024", ascending=False, na_position="last")
cas_journal_diff_csv = save_table(cas_journal_diff, cas_year_diff_dir / "CAS_presence_by_journal_percentage_point_change_2024_vs_2025.csv")

plot_journal = cas_journal_diff.dropna(subset=["percentage_point_change_2025_minus_2024"]).sort_values("percentage_point_change_2025_minus_2024")
fig, ax = plt.subplots(figsize=(11, max(5.2, 0.55 * len(plot_journal) + 1.8)))
colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in plot_journal["percentage_point_change_2025_minus_2024"]]
ax.barh(plot_journal["journal"], plot_journal["percentage_point_change_2025_minus_2024"], color=colors, height=0.65)
ax.axvline(0, color="#333333", linewidth=0.9)
max_abs = max(abs(plot_journal["percentage_point_change_2025_minus_2024"]).max(), 1)
ax.set_xlim(-max_abs * 1.25, max_abs * 1.25)
ax.set_xlabel("Percentage-point change in CAS presence, 2025 minus 2024")
ax.set_title("CAS presence by journal: percentage-point change", loc="left", fontsize=13, fontweight="600")
for row in plot_journal.itertuples(index=False):
    value = row.percentage_point_change_2025_minus_2024
    ha = "left" if value >= 0 else "right"
    offset = max_abs * 0.035 if value >= 0 else -max_abs * 0.035
    ax.text(value + offset, row.journal, f"{value:+.2f} pp", va="center", ha=ha, fontsize=8.5)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
cas_journal_diff_png = cas_year_diff_dir / "CAS_presence_by_journal_percentage_point_change_2024_vs_2025.png"
fig.savefig(cas_journal_diff_png, dpi=180, bbox_inches="tight")
plt.close(fig)

def save_cas_category_year_diff(
    label_col: str,
    order: list[str],
    display_name: str,
    subset: pd.DataFrame | None = None,
    filename_prefix: str | None = None,
    title: str | None = None,
) -> tuple[Path, Path]:
    source = cas_final if subset is None else subset
    summary_2024 = count_summary(source[source["publication_year"].eq("2024")], label_col, order)
    summary_2025 = count_summary(source[source["publication_year"].eq("2025")], label_col, order)
    diff = summary_2024.merge(summary_2025, on="label", how="outer", suffixes=("_2024", "_2025")).fillna(0)
    diff["label"] = pd.Categorical(diff["label"], categories=order, ordered=True)
    diff = diff.sort_values("label")
    for col in ["count_2024", "n_total_2024", "count_2025", "n_total_2025"]:
        diff[col] = diff[col].astype(int)
    diff["pct_2024"] = diff["pct_2024"].astype(float)
    diff["pct_2025"] = diff["pct_2025"].astype(float)
    diff["count_change_2025_minus_2024"] = diff["count_2025"] - diff["count_2024"]
    diff["percentage_point_change_2025_minus_2024"] = diff["pct_2025"] - diff["pct_2024"]
    diff["relative_percent_change_2025_vs_2024"] = np.where(diff["pct_2024"].ne(0), diff["percentage_point_change_2025_minus_2024"] / diff["pct_2024"] * 100, np.nan)
    file_stem = filename_prefix or f"CAS_{display_name}_classification_percentage_point_change_2024_vs_2025"
    csv_path = save_table(diff, cas_year_diff_dir / f"{file_stem}.csv")
    plot_diff = diff.sort_values("percentage_point_change_2025_minus_2024")
    fig, ax = plt.subplots(figsize=(9.5, max(4.8, 0.55 * len(plot_diff) + 1.8)))
    bar_colors = ["#1D9E75" if value >= 0 else "#D95F02" for value in plot_diff["percentage_point_change_2025_minus_2024"]]
    ax.barh(plot_diff["label"], plot_diff["percentage_point_change_2025_minus_2024"], color=bar_colors, height=0.65)
    ax.axvline(0, color="#333333", linewidth=0.9)
    max_abs = max(abs(plot_diff["percentage_point_change_2025_minus_2024"]).max(), 1)
    ax.set_xlim(-max_abs * 1.35, max_abs * 1.35)
    ax.set_xlabel("Percentage-point change in category share, 2025 minus 2024")
    ax.set_title(title or f"CAS {display_name} category-share changes", loc="left", fontsize=13, fontweight="600")
    for row in plot_diff.itertuples(index=False):
        value = row.percentage_point_change_2025_minus_2024
        ha = "left" if value >= 0 else "right"
        offset = max_abs * 0.045 if value >= 0 else -max_abs * 0.045
        ax.text(value + offset, row.label, f"{value:+.2f} pp", va="center", ha=ha, fontsize=9)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
    ax.set_axisbelow(True)
    fig.tight_layout()
    png_path = cas_year_diff_dir / f"{file_stem}.png"
    fig.savefig(png_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return csv_path, png_path

mca_diff_csv, mca_diff_png = save_cas_category_year_diff("MCA_label", MCA_ORDER, "MCA")
eta_diff_csv, eta_diff_png = save_cas_category_year_diff("ETA_label", ETA_ORDER, "ETA")

cas_mca_access_conditions = cas_final[cas_final["MCA_label"].isin(mca_access_condition_labels)].copy()
mca_access_condition_diff_csv, mca_access_condition_diff_png = save_cas_category_year_diff(
    "MCA_label",
    mca_access_condition_labels,
    "MCA access-condition",
    subset=cas_mca_access_conditions,
    filename_prefix="CAS_MCA_access_condition_category_share_percentage_point_change_2024_vs_2025",
    title="CAS MCA access-condition category-share changes",
)

cas_outputs.extend([
    cas_presence_year_diff_csv,
    cas_presence_year_diff_png,
    cas_presence_delta_png,
    cas_journal_diff_csv,
    cas_journal_diff_png,
    mca_diff_csv,
    mca_diff_png,
    eta_diff_csv,
    eta_diff_png,
    mca_access_condition_diff_csv,
    mca_access_condition_diff_png,
])

print(f"Resolved CAS classification rows: {len(cas_final):,}")
print(f"Unique CAS-classified publications: {cas_final['doc_doi'].nunique():,}")
print("Saved CAS outputs:")
for path in cas_outputs:
    print(path)

display(cas_final["publication_year"].value_counts().sort_index().rename("CAS classified records"))
display(count_summary(cas_final, "MCA_label", MCA_ORDER))
display(count_summary(cas_final, "ETA_label", ETA_ORDER))


/tmp/ipykernel_7152/426238779.py:333: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


Resolved CAS classification rows: 5,303
Unique CAS-classified publications: 5,303
Saved CAS outputs:
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/CAS_presence_2024_scientific_studies.csv
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/CAS_presence_2024_scientific_studies.png
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/CAS_presence_by_journal_2024_scientific_studies.csv
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/CAS_presence_by_journal_2024_scientific_studies.png
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/MCA_classification_2024.csv
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/MCA_classification_2024.png
/home/jzupp/projects/medialab_internship/data/images/CAS/final_run_2026-05-08/2024/MCA_classification_excluding_no_code_2024.csv
/home/jzupp/projects/medialab_

publication_year
2024    1302
2025    4001
Name: CAS classified records, dtype: int64

,label,count,pct,n_total
0,no_code,4207,79.332453,5303
1,restricted_access,583,10.993777,5303
2,open_access,340,6.411465,5303
3,unclear,92,1.734867,5303
4,partial_access,18,0.339431,5303
5,no_access,28,0.528003,5303
6,incorrect,35,0.660004,5303


,label,count,pct,n_total
0,no_tool,5097,96.115406,5303
1,proprietary_tool,39,0.735433,5303
2,open_tool,136,2.564586,5303
3,mixed_tool,29,0.546860,5303
4,unknown_status_tool,2,0.037715,5303


## Section Classification Coverage

Unique DAS/CAS section coverage compared with unique sections that received classification in the finalized 2024-2025 scientific-study sample.

In [5]:

# Unique section/classification coverage for finalized 2024-2025 scientific-study sample.
section_sample = df_document_registry[
    df_document_registry["publication_year"].isin(SAMPLE_YEARS)
    & df_document_registry["doc_type"].eq("Scientific study")
].copy()

unique_das_section_dois = set(section_sample.loc[bool_true(section_sample["has_DAS"]), "doc_doi"])
unique_cas_section_dois = set(section_sample.loc[bool_true(section_sample["has_CAS"]), "doc_doi"])
unique_das_classified_dois = set(das_final["doc_doi"])
unique_cas_classified_dois = set(cas_final["doc_doi"])

section_coverage = pd.DataFrame([
    {
        "section_type": "DAS",
        "unique_sections": len(unique_das_section_dois),
        "unique_classified_sections": len(unique_das_classified_dois),
        "missing_classifications": len(unique_das_section_dois - unique_das_classified_dois),
        "classification_coverage_pct": len(unique_das_classified_dois) / len(unique_das_section_dois) * 100,
        "missing_dois": "; ".join(sorted(unique_das_section_dois - unique_das_classified_dois)),
    },
    {
        "section_type": "CAS",
        "unique_sections": len(unique_cas_section_dois),
        "unique_classified_sections": len(unique_cas_classified_dois),
        "missing_classifications": len(unique_cas_section_dois - unique_cas_classified_dois),
        "classification_coverage_pct": len(unique_cas_classified_dois) / len(unique_cas_section_dois) * 100,
        "missing_dois": "; ".join(sorted(unique_cas_section_dois - unique_cas_classified_dois)),
    },
])
section_coverage.loc["Total", ["section_type", "unique_sections", "unique_classified_sections", "missing_classifications", "classification_coverage_pct", "missing_dois"]] = [
    "Total",
    section_coverage["unique_sections"].sum(),
    section_coverage["unique_classified_sections"].sum(),
    section_coverage["missing_classifications"].sum(),
    section_coverage["unique_classified_sections"].sum() / section_coverage["unique_sections"].sum() * 100,
    "; ".join(filter(None, section_coverage["missing_dois"].tolist())),
]
section_coverage = section_coverage.reset_index(drop=True)

coverage_path = save_table(section_coverage, IMAGES_DIR / "samples" / f"section_classification_coverage_{RUN_STAMP}.csv")
print(f"Section classification coverage table written to: {coverage_path}")
display(section_coverage)


Section classification coverage table written to: /home/jzupp/projects/medialab_internship/data/images/samples/section_classification_coverage_2026-05-08_123024.csv


,section_type,unique_sections,unique_classified_sections,missing_classifications,classification_coverage_pct,missing_dois
0,DAS,10844.0,10843.0,1.0,99.990778,10.1093/ptep/ptaf057
1,CAS,5304.0,5303.0,1.0,99.981146,10.1016/j.physletb.2024.138800
2,Total,16148.0,16146.0,2.0,99.987615,10.1093/ptep/ptaf057; 10.1016/j.physletb.2024....


## Manifest

In [6]:

# Final-run manifest for thesis freeze outputs.
manifest = {
    "run_date": RUN_DATE,
    "run_stamp": RUN_STAMP,
    "scoap_totals": SCOAP_TOTALS,
    "sample_dir": str(SAMPLE_DIR),
    "das_final_dir": str(DAS_FINAL_DIR),
    "cas_final_dir": str(CAS_FINAL_DIR),
    "previous_das_final_dir": str(PREVIOUS_DAS_FINAL_DIR),
    "previous_cas_final_dir": str(PREVIOUS_CAS_FINAL_DIR),
    "sample_files": sorted(str(path.relative_to(PROJECT_ROOT)) for path in SAMPLE_DIR.glob(f"*{RUN_STAMP}*")),
    "das_files": sorted(str(path.relative_to(PROJECT_ROOT)) for path in DAS_FINAL_DIR.rglob("*" ) if path.is_file()),
    "cas_files": sorted(str(path.relative_to(PROJECT_ROOT)) for path in CAS_FINAL_DIR.rglob("*" ) if path.is_file()),
}
manifest_path = IMAGES_DIR / f"descriptive_statistics_final_manifest_{RUN_STAMP}.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest written to: {manifest_path}")
print(f"Sample files generated: {len(manifest['sample_files'])}")
print(f"DAS files generated: {len(manifest['das_files'])}")
print(f"CAS files generated: {len(manifest['cas_files'])}")


Manifest written to: /home/jzupp/projects/medialab_internship/data/images/descriptive_statistics_final_manifest_2026-05-08_014103.json
Sample files generated: 5
DAS files generated: 24
CAS files generated: 42
